# Traditional ML Classification: Broken vs Complete 3D Objects

Binary classification using **handcrafted 3D geometric features** extracted from PLY meshes,
trained with classical ML algorithms. Uses **Open3D CUDA** for GPU-accelerated mesh processing.

**Feature Categories:**
1. Curvature statistics
2. Surface area / Volume ratios
3. Normal vector statistics
4. Edge / Boundary analysis
5. Shape descriptors (PCA, bounding box)

**ML Classifiers:**
Random Forest, SVM (RBF), XGBoost, k-NN, Gradient Boosting

## 1. Setup & Imports

In [ ]:
import os, sys, glob, time, warnings
import numpy as np
import open3d as o3d
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import joblib

try:
    import xgboost as xgb
    HAS_XGBOOST = True
    print("XGBoost:", xgb.__version__)
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not available")

warnings.filterwarnings('ignore')

# ── Verify Open3D CUDA ──
print(f"Open3D: {o3d.__version__}")
try:
    t = o3d.core.Tensor([1,2,3], device=o3d.core.Device('CUDA:0'))
    print(f"Open3D CUDA: Available (device={t.device})")
    USE_CUDA = True
except Exception as e:
    print(f"Open3D CUDA: Not available ({e}), using CPU")
    USE_CUDA = False

## 2. Configuration

In [ ]:
DATA_ROOT   = 'data/Fantastic_Breaks_v1'
OUTPUT_DIR  = 'results/traditional_ml'
NUM_SAMPLE  = None     # None = use ALL mesh vertices (no subsampling)
N_FOLDS     = 5        # cross-validation folds
SEED        = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Data Discovery

In [ ]:
def discover_meshes(data_root):
    """Find complete (model_c.ply) and broken (model_b_0.ply) meshes."""
    samples = []
    for cat_dir in sorted(glob.glob(os.path.join(data_root, '*'))):
        if not os.path.isdir(cat_dir): continue
        for obj_dir in sorted(glob.glob(os.path.join(cat_dir, '*'))):
            if not os.path.isdir(obj_dir): continue
            obj_id = os.path.basename(obj_dir)
            c_path = os.path.join(obj_dir, 'model_c.ply')
            b_path = os.path.join(obj_dir, 'model_b_0.ply')
            if os.path.exists(c_path):
                samples.append({'path': c_path, 'label': 0, 'name': 'complete', 'id': f'{obj_id}_c'})
            if os.path.exists(b_path):
                samples.append({'path': b_path, 'label': 1, 'name': 'broken',   'id': f'{obj_id}_b'})
    return samples

samples = discover_meshes(DATA_ROOT)
n_c = sum(1 for s in samples if s['label'] == 0)
n_b = sum(1 for s in samples if s['label'] == 1)
print(f"Found {len(samples)} meshes: {n_c} complete + {n_b} broken")

## 4. Feature Extraction Functions

Each function computes one category of geometric features from a mesh.
Open3D CUDA is used for point cloud operations and KD-tree queries where supported.

### 4.1 Curvature Statistics

In [ ]:
def compute_curvature_stats(mesh, pcd):
    """
    Estimate discrete surface curvature via PCA on local neighborhoods.
    For each sample point, fit a local PCA and compute surface variation
    (ratio of smallest eigenvalue to sum) as a curvature proxy.
    """
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    points = np.asarray(pcd.points)
    n_points = len(points)

    # Sample subset for speed
    rng = np.random.default_rng(42)
    sample_idx = np.arange(n_points)  # use ALL points
    curvatures = np.zeros(n_points, dtype=np.float64)

    for i, idx in enumerate(sample_idx):
        k = min(30, n_points)
        [_, nn_idx, _] = pcd_tree.search_knn_vector_3d(points[idx], k)
        if len(nn_idx) < 3: continue

        # PCA on local neighborhood → surface variation
        neighbors = points[nn_idx]
        centered = neighbors - neighbors.mean(axis=0)
        cov = centered.T @ centered / len(neighbors)
        eigenvalues = np.sort(np.abs(np.linalg.eigvalsh(cov)))
        total = eigenvalues.sum()
        if total > 0:
            curvatures[i] = eigenvalues[0] / total  # surface variation

    return {
        'curv_min': float(np.min(curvatures)),
        'curv_max': float(np.max(curvatures)),
        'curv_mean': float(np.mean(curvatures)),
        'curv_std': float(np.std(curvatures)),
        'curv_median': float(np.median(curvatures)),
        'curv_q25': float(np.percentile(curvatures, 25)),
        'curv_q75': float(np.percentile(curvatures, 75)),
    }

### 4.2 Surface Area / Volume

In [ ]:
def compute_surface_volume_stats(mesh):
    """
    Surface area, convex hull volume, SA/V ratio, convexity, sphericity.
    Convexity = mesh_area / hull_area measures how 'convex' the shape is.
    Broken objects often have higher convexity (more surface detail).
    """
    surface_area = mesh.get_surface_area()
    vertices = np.asarray(mesh.vertices)

    try:
        hull = ConvexHull(vertices)
        volume = hull.volume
        convex_area = hull.area
    except:
        volume, convex_area = 0.0, surface_area

    convexity = surface_area / convex_area if convex_area > 0 else 1.0
    if surface_area > 0 and volume > 0:
        sphericity = (np.pi**(1/3) * (6*volume)**(2/3)) / surface_area
    else:
        sphericity = 0.0
    sa_v_ratio = surface_area / volume if volume > 0 else 0.0

    return {
        'surface_area': float(surface_area),
        'volume': float(volume),
        'sa_v_ratio': float(sa_v_ratio),
        'convexity': float(convexity),
        'sphericity': float(sphericity),
    }

### 4.3 Normal Statistics

In [ ]:
def compute_normal_stats(pcd):
    """
    Normal direction variance, entropy, and consistency.
    Normal consistency = average alignment of neighboring normals.
    Broken surfaces have lower consistency (more irregular normals).
    """
    normals = np.asarray(pcd.normals)
    if len(normals) == 0:
        return {k: 0.0 for k in ['normal_var_x','normal_var_y','normal_var_z',
                                   'normal_var_total','normal_entropy','normal_consistency']}

    var_x, var_y, var_z = [float(np.var(normals[:,i])) for i in range(3)]
    var_total = var_x + var_y + var_z

    # Normal consistency via KNN
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    n_points = len(normals)
    rng = np.random.default_rng(42)
    sample_idx = np.arange(n_points)  # use ALL points
    consistencies = []
    for idx in sample_idx:
        [_, nn_idx, _] = pcd_tree.search_knn_vector_3d(np.asarray(pcd.points)[idx], min(15, n_points))
        if len(nn_idx) < 2: continue
        dots = np.abs(normals[nn_idx[1:]] @ normals[idx])
        consistencies.append(np.mean(dots))
    normal_consistency = float(np.mean(consistencies)) if consistencies else 0.0

    # Normal direction entropy (histogram on unit sphere)
    theta = np.arccos(np.clip(normals[:,2], -1, 1))
    phi = np.arctan2(normals[:,1], normals[:,0])
    hist, _, _ = np.histogram2d(theta, phi, bins=20)
    hist = hist / hist.sum()
    hist = hist[hist > 0]
    normal_entropy = float(-np.sum(hist * np.log2(hist)))

    return {'normal_var_x': var_x, 'normal_var_y': var_y, 'normal_var_z': var_z,
            'normal_var_total': var_total, 'normal_entropy': normal_entropy,
            'normal_consistency': normal_consistency}

### 4.4 Edge / Boundary Analysis

In [ ]:
def compute_edge_boundary_stats(mesh):
    """
    Boundary edges, non-manifold edges/vertices, Euler characteristic, watertightness.
    Broken objects typically have more boundary edges (open fracture surfaces).
    """
    non_manifold_edges = len(mesh.get_non_manifold_edges(allow_boundary_edges=False))
    all_nm_edges = len(mesh.get_non_manifold_edges(allow_boundary_edges=True))
    n_boundary = all_nm_edges - non_manifold_edges
    n_nm_vertices = len(mesh.get_non_manifold_vertices())

    # Count total edges from triangles
    triangles = np.asarray(mesh.triangles)
    edges = set()
    for tri in triangles:
        for j in range(3):
            edges.add(tuple(sorted([tri[j], tri[(j+1)%3]])))
    n_total_edges = len(edges)

    boundary_ratio = n_boundary / n_total_edges if n_total_edges > 0 else 0.0
    n_v, n_f = len(mesh.vertices), len(mesh.triangles)
    euler = n_v - n_total_edges + n_f

    return {
        'n_boundary_edges': float(n_boundary),
        'boundary_ratio': float(boundary_ratio),
        'n_non_manifold_edges': float(non_manifold_edges),
        'n_non_manifold_vertices': float(n_nm_vertices),
        'euler_characteristic': float(euler),
        'is_watertight': float(mesh.is_watertight()),
        'n_vertices': float(n_v),
        'n_faces': float(n_f),
    }

### 4.5 Shape Descriptors

In [ ]:
def compute_shape_descriptors(mesh):
    """
    PCA eigenvalue ratios, elongation, flatness, bounding box aspect ratios.
    Captures the overall shape distribution of the object.
    """
    vertices = np.asarray(mesh.vertices)
    centered = vertices - vertices.mean(axis=0)
    cov = centered.T @ centered / len(vertices)
    eigenvalues = np.sort(np.linalg.eigvalsh(cov))[::-1]

    ev_sum = eigenvalues.sum()
    if ev_sum > 0:
        ev_ratios = eigenvalues / ev_sum
    else:
        ev_ratios = np.zeros(3)

    elongation = eigenvalues[1] / eigenvalues[0] if eigenvalues[0] > 0 else 0.0
    flatness   = eigenvalues[2] / eigenvalues[0] if eigenvalues[0] > 0 else 0.0

    try:
        obb = mesh.get_oriented_bounding_box()
        extent = np.sort(obb.extent)[::-1]
        bb_r1 = extent[1]/extent[0] if extent[0] > 0 else 0.0
        bb_r2 = extent[2]/extent[0] if extent[0] > 0 else 0.0
        bb_vol = float(np.prod(extent))
    except:
        bb_r1, bb_r2, bb_vol = 0.0, 0.0, 0.0

    return {
        'pca_ev_ratio_1': float(ev_ratios[0]),
        'pca_ev_ratio_2': float(ev_ratios[1]),
        'pca_ev_ratio_3': float(ev_ratios[2]),
        'elongation': float(elongation),
        'flatness': float(flatness),
        'bb_aspect_ratio_1': float(bb_r1),
        'bb_aspect_ratio_2': float(bb_r2),
        'bb_volume': float(bb_vol),
    }

### 4.6 Combined Feature Extraction

In [ ]:
def extract_features(mesh_path, num_sample_points=None):
    """Extract all 34 geometric features from a single PLY mesh.
    If num_sample_points is None, uses ALL mesh vertices (no subsampling).
    """
    mesh = o3d.io.read_triangle_mesh(mesh_path)
    mesh.compute_vertex_normals()

    # Use all vertices or sample a subset
    if num_sample_points is None:
        pcd = o3d.geometry.PointCloud()
        pcd.points = mesh.vertices  # use ALL vertices
    else:
        pcd = mesh.sample_points_uniformly(number_of_points=num_sample_points)
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))

    features = {}
    features.update(compute_curvature_stats(mesh, pcd))
    features.update(compute_surface_volume_stats(mesh))
    features.update(compute_normal_stats(pcd))
    features.update(compute_edge_boundary_stats(mesh))
    features.update(compute_shape_descriptors(mesh))
    return features

print(f"Feature extraction pipeline ready (sample={NUM_SAMPLE or 'ALL vertices'}, CUDA: {USE_CUDA})")

## 5. Extract Features from All Meshes

In [ ]:
all_features, all_labels, all_ids, failed = [], [], [], []
t0 = time.time()

for i, sample in enumerate(samples):
    if (i+1) % 10 == 0 or i == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(samples) - i - 1)
        print(f"  [{i+1:3d}/{len(samples)}] {sample['id']:15s}  "
              f"({elapsed:.0f}s elapsed, ~{eta:.0f}s ETA)")
    try:
        feats = extract_features(sample['path'], NUM_SAMPLE)
        all_features.append(feats)
        all_labels.append(sample['label'])
        all_ids.append(sample['id'])
    except Exception as e:
        failed.append((sample['id'], str(e)))
        print(f"  WARNING: {sample['id']}: {e}")

t_total = time.time() - t0
print(f"\nDone: {len(all_features)} samples in {t_total:.1f}s ({t_total/len(all_features):.1f}s/mesh)")
if failed: print(f"Failed: {len(failed)}")

# Convert to arrays
feature_names = list(all_features[0].keys())
X = np.nan_to_num(np.array([[f[k] for k in feature_names] for f in all_features], dtype=np.float64))
y = np.array(all_labels, dtype=np.int32)

# Cache features for later reuse
np.savez(os.path.join(OUTPUT_DIR, 'features.npz'), X=X, y=y,
         feature_names=feature_names, object_ids=all_ids)
print(f"Feature matrix: {X.shape} — saved to {OUTPUT_DIR}/features.npz")

### (Optional) Load Cached Features
Run this cell to skip re-extraction if features are already saved.

In [ ]:
# Uncomment to load from cache instead of re-running extraction:
# data = np.load(os.path.join(OUTPUT_DIR, 'features.npz'), allow_pickle=True)
# X = data['X']; y = data['y']
# feature_names = list(data['feature_names']); all_ids = list(data['object_ids'])
# print(f"Loaded cached features: {X.shape}")

## 6. Feature Analysis & Visualization

In [ ]:
# ── Feature distribution comparison ──
fig, axes = plt.subplots(6, 6, figsize=(24, 20))
axes = axes.flatten()

for i, fname in enumerate(feature_names):
    if i >= len(axes): break
    ax = axes[i]
    ax.hist(X[y==0, i], bins=20, alpha=0.6, label='Complete', color='steelblue', density=True)
    ax.hist(X[y==1, i], bins=20, alpha=0.6, label='Broken', color='coral', density=True)
    ax.set_title(fname, fontsize=9)
    ax.tick_params(labelsize=7)
    if i == 0: ax.legend(fontsize=7)

# Hide unused subplots
for j in range(len(feature_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Complete vs Broken', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Training & Cross-Validation

We use **5-fold stratified cross-validation** to evaluate each classifier.
All features are standardized (zero mean, unit variance) before training.

In [ ]:
# ── Define classifiers ──
classifiers = {
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=5,
        random_state=SEED, n_jobs=-1),
    'SVM_RBF': SVC(kernel='rbf', C=10.0, gamma='scale',
                   probability=True, random_state=SEED),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=SEED),
}
if HAS_XGBOOST:
    classifiers['XGBoost'] = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        use_label_encoder=False, eval_metric='logloss',
        random_state=SEED, n_jobs=-1, tree_method='hist')

# ── Scale features ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ── Cross-validation ──
results = {}
for name, clf in classifiers.items():
    print(f"\n--- {name} ---")
    y_pred = cross_val_predict(clf, X_scaled, y, cv=cv, method='predict')
    try:
        y_prob = cross_val_predict(clf, X_scaled, y, cv=cv, method='predict_proba')[:,1]
        auc = roc_auc_score(y, y_prob)
    except:
        y_prob, auc = None, None

    acc  = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec  = recall_score(y, y_pred)
    f1   = f1_score(y, y_pred)
    cm   = confusion_matrix(y, y_pred)

    results[name] = dict(accuracy=acc, precision=prec, recall=rec, f1=f1,
                         roc_auc=auc, cm=cm, y_true=y, y_pred=y_pred, y_prob=y_prob)
    print(f"  Acc={acc:.4f}  Prec={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}", end="")
    print(f"  AUC={auc:.4f}" if auc else "")

## 8. Results Summary

In [ ]:
# ── Summary table ──
print(f"{'Classifier':<20} {'Acc':>8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'AUC':>8}")
print("-" * 60)
for name, r in results.items():
    auc_s = f"{r['roc_auc']:.4f}" if r['roc_auc'] else "  N/A"
    print(f"{name:<20} {r['accuracy']:>8.4f} {r['precision']:>8.4f} "
          f"{r['recall']:>8.4f} {r['f1']:>8.4f} {auc_s:>8}")

# ── Classification reports ──
for name, r in results.items():
    print(f"\n{'='*40}\n{name}\n{'='*40}")
    print(classification_report(r['y_true'], r['y_pred'], target_names=['Complete','Broken']))

## 9. Confusion Matrices

In [ ]:
n_clf = len(results)
fig, axes = plt.subplots(1, n_clf, figsize=(5*n_clf, 4))
if n_clf == 1: axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    cm = r['cm']
    im = ax.imshow(cm, cmap=plt.cm.Blues)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Complete','Broken']); ax.set_yticklabels(['Complete','Broken'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(name)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

## 10. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, r in results.items():
    if r['roc_auc'] is not None:
        fpr, tpr, _ = roc_curve(r['y_true'], r['y_prob'])
        ax.plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})")
ax.plot([0,1],[0,1],'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Broken vs Complete'); ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curves.png'), dpi=150)
plt.show()

## 11. Feature Importance (Random Forest)

In [ ]:
# Train RF on full data for feature importance
rf = classifiers['RandomForest']
rf.fit(X_scaled, y)
importances = rf.feature_importances_
imp_idx = np.argsort(importances)[::-1]

print("Top 15 features:")
for i in range(min(15, len(feature_names))):
    print(f"  {i+1:2d}. {feature_names[imp_idx[i]]:30s}  {importances[imp_idx[i]]:.4f}")

# Plot
top_n = min(15, len(feature_names))
idx = imp_idx[:top_n]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_n), importances[idx][::-1], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in idx][::-1])
ax.set_xlabel('Feature Importance')
ax.set_title('Top Features (Random Forest)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150)
plt.show()

## 12. Save Trained Models

In [ ]:
# Train all classifiers on full data and save
for name, clf in classifiers.items():
    clf.fit(X_scaled, y)
    path = os.path.join(OUTPUT_DIR, f'{name}_model.joblib')
    joblib.dump(clf, path)
    print(f"Saved: {path}")

joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.joblib'))
print(f"Saved: {OUTPUT_DIR}/scaler.joblib")

## 13. Inference on New Mesh

Load a saved model and predict on a new PLY file.

In [ ]:
def predict_mesh(mesh_path, model_path=None, scaler_path=None):
    """Predict whether a mesh is broken or complete."""
    # Default paths
    if model_path is None:
        model_path = os.path.join(OUTPUT_DIR, 'RandomForest_model.joblib')
    if scaler_path is None:
        scaler_path = os.path.join(OUTPUT_DIR, 'scaler.joblib')

    # Extract features
    feats = extract_features(mesh_path, NUM_SAMPLE)
    X_new = np.array([[feats[k] for k in feature_names]])
    X_new = np.nan_to_num(X_new)

    # Load model and predict
    model = joblib.load(model_path)
    sc = joblib.load(scaler_path)
    X_new_scaled = sc.transform(X_new)

    pred = model.predict(X_new_scaled)[0]
    prob = model.predict_proba(X_new_scaled)[0]

    label_names = ['Complete', 'Broken']
    print(f"Mesh: {mesh_path}")
    print(f"Prediction: {label_names[pred]}")
    print(f"Confidence: Complete={prob[0]:.3f}, Broken={prob[1]:.3f}")
    return pred, prob

# ── Example inference ──
test_mesh = samples[0]['path']         # first complete mesh
print(f"True label: {samples[0]['name']}\n")
predict_mesh(test_mesh)

print("\n" + "="*40 + "\n")

test_mesh = samples[1]['path']         # first broken mesh
print(f"True label: {samples[1]['name']}\n")
predict_mesh(test_mesh)